In [1]:
import pandas as pd
import numpy as np

# 重新读取数据
df = pd.read_excel('附件1.xlsx')

# 提取周数据列
week_columns = [col for col in df.columns if col.startswith('W')]
df_week = df[week_columns].copy()
df_week = df_week.apply(pd.to_numeric, errors='coerce')

# 按24周划分周期（10个周期）
n_periods = 10
period_weeks = 24
total_weeks = n_periods * period_weeks

print(f"周期划分: {n_periods}个周期 × {period_weeks}周 = {total_weeks}周")

# 为每个供应商计算期望供货量
expected_supply = []

for supplier_idx in range(len(df)):
    supplier_id = df.iloc[supplier_idx]['供应商ID']
    material_type = df.iloc[supplier_idx]['材料分类']

    # 获取该供应商的240周数据
    supplier_data = df_week.iloc[supplier_idx].values

    # 按周期分组（10个周期，每个周期24周）
    periods_data = []
    for period in range(n_periods):
        start_idx = period * period_weeks
        end_idx = start_idx + period_weeks
        period_data = supplier_data[start_idx:end_idx]
        periods_data.append(period_data)

    # 计算期望供货量
    weekly_expected = []
    for week_in_period in range(period_weeks):
        # 提取10个周期中对应周次的数据
        week_data = [periods_data[period][week_in_period] for period in range(n_periods)]

        # 移除NaN值
        week_data_clean = [x for x in week_data if pd.notna(x)]

        if not week_data_clean:
            weekly_expected.append(0)
            continue

        # 计算该周次的均值
        week_mean = np.mean(week_data_clean)

        # 筛选条件：供货量/订货量 > 90%（假设订货量=供货量，所以都满足）且 供货量 ≤ 3倍均值
        # 由于假设订货量=供货量，所以供货量/订货量 = 100% > 90%
        valid_data = [x for x in week_data_clean if x <= 3 * week_mean]

        if not valid_data:
            weekly_expected.append(0)
        else:
            # 取最大值作为期望供货量
            weekly_expected.append(max(valid_data))

    # 存储结果
    expected_supply.append({
        '供应商ID': supplier_id,
        '材料分类': material_type,
        **{f'未来第{i + 1}周': weekly_expected[i] for i in range(period_weeks)}
    })

# 创建结果DataFrame
result_df = pd.DataFrame(expected_supply)

print(f"\n期望供货量计算完成，共{len(result_df)}个供应商")
print("\n前5个供应商的期望供货量（前10周）:")
display_cols = ['供应商ID', '材料分类'] + [f'未来第{i + 1}周' for i in range(10)]
print(result_df[display_cols].head())

# 保存结果
result_df.to_excel('期望供货量预测结果.xlsx', index=False)
print(f"\n结果已保存到: 期望供货量预测结果.xlsx")

# 统计信息
print("\n期望供货量统计信息:")
future_weeks = [col for col in result_df.columns if col.startswith('未来第')]
future_data = result_df[future_weeks]
print(f"各周期望供货量平均值:")
for week in future_weeks[:10]:  # 显示前10周
    print(f"{week}: {future_data[week].mean():.2f}")

周期划分: 10个周期 × 24周 = 240周

期望供货量计算完成，共403个供应商

前5个供应商的期望供货量（前10周）:
  供应商ID 材料分类  未来第1周  未来第2周  未来第3周  未来第4周  未来第5周  未来第6周  未来第7周  未来第8周  未来第9周  \
0  S001    B    0.0    0.0    1.0    1.0    0.0    0.0    0.0    1.0    1.0   
1  S002    A    1.0    1.0    1.0    2.0    1.0    0.0    0.0    1.0    1.0   
2  S003    C   10.0    3.0    0.0    0.0    1.0    0.0    4.0   70.0  440.0   
3  S004    B    1.0    1.0    1.0    1.0    1.0    1.0    0.0    1.0    1.0   
4  S005    A   30.0    0.0    1.0   60.0    1.0   70.0   60.0   60.0    0.0   

   未来第10周  
0     1.0  
1     3.0  
2   380.0  
3     1.0  
4     1.0  

结果已保存到: 期望供货量预测结果.xlsx

期望供货量统计信息:
各周期望供货量平均值:
未来第1周: 137.75
未来第2周: 59.49
未来第3周: 70.49
未来第4周: 51.92
未来第5周: 104.21
未来第6周: 68.22
未来第7周: 49.08
未来第8周: 61.41
未来第9周: 63.58
未来第10周: 58.89


In [2]:
import pandas as pd
import numpy as np
import json

# 读取结果数据
result_df = pd.read_excel('期望供货量预测结果.xlsx')

# 分析结果数据
print("=== 期望供货量分析 ===")

# 1. 按材料分类统计
material_stats = {}
future_weeks = [col for col in result_df.columns if col.startswith('未来第')]

for material in ['A', 'B', 'C']:
    material_data = result_df[result_df['材料分类'] == material]
    weekly_means = [float(material_data[week].mean()) for week in future_weeks]

    material_stats[material] = {
        '供应商数量': int(len(material_data)),
        '平均期望供货量': float(np.mean(weekly_means)),
        '周平均供货量': weekly_means
    }

# 2. 周度趋势分析
weekly_totals = [float(result_df[week].sum()) for week in future_weeks]
weekly_avgs = [float(result_df[week].mean()) for week in future_weeks]

# 3. 供应商供货能力分析
supplier_totals = result_df[future_weeks].sum(axis=1)
supplier_totals_list = [float(x) for x in supplier_totals]

# 4. 供应商分布数据
supplier_ranges = ['0-100', '101-500', '501-1000', '1001-5000', '5000+']
supplier_distribution = [
    int(len(supplier_totals[supplier_totals <= 100])),
    int(len(supplier_totals[(supplier_totals > 100) & (supplier_totals <= 500)])),
    int(len(supplier_totals[(supplier_totals > 500) & (supplier_totals <= 1000)])),
    int(len(supplier_totals[(supplier_totals > 1000) & (supplier_totals <= 5000)])),
    int(len(supplier_totals[supplier_totals > 5000]))
]

# 5. 准备样例数据（前10个供应商的前10周数据）
sample_data = []
for i in range(min(10, len(result_df))):
    row = result_df.iloc[i]
    sample_record = {
        '供应商ID': str(row['供应商ID']),
        '材料分类': str(row['材料分类']),
        '周数据': [float(row[f'未来第{j + 1}周']) for j in range(10)]
    }
    sample_data.append(sample_record)

# 6. 准备可视化数据
viz_data = {
    'sample_data': sample_data,
    'weekly_trend': {
        'weeks': [f'第{i + 1}周' for i in range(24)],
        'total_supply': weekly_totals,
        'avg_supply': weekly_avgs
    },
    'material_comparison': {
        'materials': list(material_stats.keys()),
        'avg_supply': [material_stats[m]['平均期望供货量'] for m in material_stats.keys()],
        'supplier_count': [material_stats[m]['供应商数量'] for m in material_stats.keys()],
        'weekly_trends': {
            m: material_stats[m]['周平均供货量'] for m in material_stats.keys()
        }
    },
    'supplier_distribution': {
        'ranges': supplier_ranges,
        'counts': supplier_distribution
    },
    'summary_stats': {
        'total_suppliers': int(len(result_df)),
        'avg_weekly_supply': float(np.mean(weekly_totals)),
        'max_weekly_supply': float(max(weekly_totals)),
        'min_weekly_supply': float(min(weekly_totals))
    }
}

print("样例数据（前3个供应商）:")
for i, record in enumerate(sample_data[:3]):
    print(f"{i + 1}. {record['供应商ID']} ({record['材料分类']}): {record['周数据'][:5]}...")

print(f"\n周度趋势: 第1周{weekly_totals[0]:.0f}, 第24周{weekly_totals[-1]:.0f}")
print(
    f"材料分类分布: A类{material_stats['A']['供应商数量']}家, B类{material_stats['B']['供应商数量']}家, C类{material_stats['C']['供应商数量']}家")
print(f"供应商分布: {dict(zip(supplier_ranges, supplier_distribution))}")
print(
    f"总体统计: 平均周供货量{viz_data['summary_stats']['avg_weekly_supply']:.0f}, 最高{viz_data['summary_stats']['max_weekly_supply']:.0f}")

print("\n数据准备完成，准备生成HTML报告...")

=== 期望供货量分析 ===
样例数据（前3个供应商）:
1. S001 (B): [0.0, 0.0, 1.0, 1.0, 0.0]...
2. S002 (A): [1.0, 1.0, 1.0, 2.0, 1.0]...
3. S003 (C): [10.0, 3.0, 0.0, 0.0, 1.0]...

周度趋势: 第1周55512, 第24周20993
材料分类分布: A类146家, B类134家, C类122家
供应商分布: {'0-100': 334, '101-500': 10, '501-1000': 6, '1001-5000': 23, '5000+': 30}
总体统计: 平均周供货量27423, 最高55512

数据准备完成，准备生成HTML报告...


In [6]:
import pandas as pd
import numpy as np
import random
import os
from openpyxl import load_workbook

# 设置中文显示
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["SimHei", "WenQuanYi Micro Hei", "Heiti TC"]

class SupplyChainOptimizer:
    def __init__(self, data_paths, sheet_mapping=None):
        """初始化优化器，加载所有必要数据"""
        self.data_paths = data_paths
        # 设置sheet名称到材料类型的映射，如果未提供则使用默认映射
        self.sheet_mapping = sheet_mapping if sheet_mapping else {
            'A类供应商': 'A',
            'B类供应商': 'B',
            'C类供应商': 'C',
            'Sheet1': 'A',  # 默认sheet名称映射
            'Sheet2': 'B',
            'Sheet3': 'C'
        }
        self.load_data()
        self.initialize_parameters()

    def load_data(self):
        """加载所有输入数据"""
        # 加载供应商TOPSIS评价结果
        self.topsis_results = pd.read_excel(
            self.data_paths['topsis_results'],
            sheet_name=None
        )

        # 加载历史供应商数据
        self.supplier_data = pd.read_excel(
            self.data_paths['supplier_data'],
            sheet_name=None
        )

        # 加载转运商数据
        self.transporter_data = pd.read_excel(
            self.data_paths['transporter_data']
        )

        # 加载期望供货量预测结果，并确保是数值类型
        self.demand_forecast = pd.read_excel(
            self.data_paths['demand_forecast']
        )
        # 转换需求预测数据为数值类型（只转换可能包含数值的列）
        for col in self.demand_forecast.columns:
            # 跳过明显是非数值的列名
            if 'ID' in str(col) or '分类' in str(col) or '名称' in str(col):
                continue
            try:
                self.demand_forecast[col] = pd.to_numeric(self.demand_forecast[col])
            except ValueError:
                print(f"警告: 无法将列 '{col}' 转换为数值类型")

        # 加载附件A和B模板
        self.template_a = pd.read_excel(
            self.data_paths['template_a'],
            sheet_name=None,
            header=None
        )
        self.template_b = pd.read_excel(
            self.data_paths['template_b'],
            sheet_name=None,
            header=None
        )

    def initialize_parameters(self):
        """初始化模型参数"""
        # 生产参数
        self.weekly_capacity = 28200  # 每周产能(立方米)
        self.weeks = 24  # 计划周期
        self.production_weeks_per_year = 48  # 每年生产周数

        # 原材料转换系数
        self.material_conversion = {
            'A': 1/0.6,   # A类材料产能系数
            'B': 1/0.66,  # B类材料产能系数
            'C': 1/0.72   # C类材料产能系数
        }

        # 原材料成本系数 (以C类为基准)
        self.cost_coefficients = {
            'A': 1.2,   # A类比C类高20%
            'B': 1.1,   # B类比C类高10%
            'C': 1.0    # C类基准
        }

        # 库存参数
        self.min_inventory_weeks = 2  # 最少库存周数
        self.min_inventory = self.weekly_capacity * self.min_inventory_weeks  # 最少库存量

        # 转运参数
        self.transporter_capacity = 6000  # 转运商每周运输能力(立方米)

        # 遗传算法参数
        self.ga_params = {
            'population_size': 50,
            'generations': 100,
            'mutation_rate': 0.1,
            'crossover_rate': 0.8
        }

    def get_material_type(self, sheet_name):
        """将sheet名称映射为材料类型(A/B/C)"""
        # 尝试直接映射
        if sheet_name in self.sheet_mapping:
            return self.sheet_mapping[sheet_name]

        # 尝试从sheet名称中提取材料类型
        for material in ['A', 'B', 'C']:
            if material in sheet_name:
                return material

        # 如果都无法识别，返回默认值并警告
        print(f"警告: 无法识别sheet名称 '{sheet_name}' 对应的材料类型，默认使用'A'")
        return 'A'

    def select_suppliers(self):
        """
        多目标规划选择供应商：
        1. 最小化供应商数量
        2. 最大化TOPSIS评分总和
        """
        # 假设从TOPSIS结果中提取供应商评分
        topsis_scores = []
        for sheet in self.topsis_results:
            df = self.topsis_results[sheet]
            # 获取材料类型
            material_type = self.get_material_type(sheet)

            # 假设第一列是供应商ID，最后一列是TOPSIS评分
            if len(df.columns) >= 2:
                for _, row in df.iterrows():
                    try:
                        supplier_id = row[0]
                        score = row.iloc[-1]
                        topsis_scores.append((supplier_id, score, material_type))
                    except:
                        continue

        # 按评分降序排序
        topsis_scores.sort(key=lambda x: x[1], reverse=True)

        # 计算总需求(等效材料)
        total_demand = self.calculate_total_demand()

        # 累计供应商产能，直到满足总需求
        selected_suppliers = []
        cumulative_capacity = 0

        for supplier in topsis_scores:
            supplier_id, score, material_type = supplier
            # 估算供应商产能(这里使用历史数据的平均值)
            capacity = self.estimate_supplier_capacity(supplier_id, material_type)

            selected_suppliers.append({
                'id': supplier_id,
                'score': score,
                'material_type': material_type,
                'capacity': capacity
            })

            # 确保材料类型有效
            if material_type not in self.material_conversion:
                print(f"错误: 无效的材料类型 {material_type}，已跳过该供应商")
                continue

            cumulative_capacity += capacity * self.material_conversion[material_type]

            # 当累计产能超过总需求的1.2倍时停止(考虑不确定性)
            if cumulative_capacity >= total_demand * 1.2:
                break

        self.selected_suppliers = selected_suppliers
        print(f"选择的供应商数量: {len(selected_suppliers)}")
        print(f"总等效产能: {cumulative_capacity:.2f}")
        print(f"总需求: {total_demand:.2f}")

        return selected_suppliers

    def calculate_total_demand(self):
        """计算24周的总等效材料需求"""
        # 每周产能对应的等效材料需求
        weekly_equivalent_demand = self.weekly_capacity

        # 24周总需求
        total_demand = weekly_equivalent_demand * self.weeks
        return total_demand

    def estimate_supplier_capacity(self, supplier_id, material_type):
        """根据历史数据估算供应商产能"""
        # 在实际应用中，这里应该根据历史数据计算供应商的平均供货能力
        # 这里简化处理，假设从供应商数据中查找
        for sheet in self.supplier_data:
            df = self.supplier_data[sheet]
            if supplier_id in df.values:
                # 找到该供应商的历史数据
                supplier_rows = df[df.iloc[:, 0] == supplier_id]
                if not supplier_rows.empty:
                    # 计算平均周供货量
                    supply_columns = [col for col in df.columns if '供货量' in str(col)]
                    if supply_columns:
                        avg_supply = supplier_rows[supply_columns].mean().mean()
                        return avg_supply
        # 如果找不到，返回一个默认值
        return 1000

    def optimize_order_plan(self):
        """优化24周的订购计划"""
        # 初始化库存
        current_inventory = self.min_inventory  # 初始库存满足两周需求
        order_plan = []

        # 按周优化
        for week in range(1, self.weeks + 1):
            # 获取本周需求预测
            week_demand = self.get_weekly_demand(week)

            # 确保需求是数值类型
            if not isinstance(week_demand, (int, float)):
                try:
                    week_demand = float(week_demand)
                except (ValueError, TypeError):
                    print(f"警告: 第{week}周的需求不是有效的数值，使用默认产能值")
                    week_demand = self.weekly_capacity

            # 应用遗传算法优化本周订购计划
            weekly_orders = self.genetic_algorithm_optimize_orders(
                week, current_inventory, week_demand
            )

            order_plan.append({
                'week': week,
                'orders': weekly_orders,
                'starting_inventory': current_inventory,
                'demand': week_demand
            })

            # 计算本周结束时的库存
            total_supply = sum(
                order['quantity'] * self.material_conversion[order['material_type']]
                for order in weekly_orders
                if order['material_type'] in self.material_conversion
            )
            current_inventory = max(
                self.min_inventory,
                current_inventory + total_supply - week_demand
            )

        self.order_plan = order_plan
        return order_plan

    def get_weekly_demand(self, week):
        """获取指定周的需求，确保返回数值类型"""
        try:
            # 从预测结果中获取，或根据产能计算
            if week <= len(self.demand_forecast):
                # 假设第一列是周数，第二列是需求
                demand_value = self.demand_forecast.iloc[week-1, 1]

                # 确保返回数值类型
                if isinstance(demand_value, (int, float)):
                    return demand_value
                else:
                    return float(demand_value)
            else:
                # 如果没有预测数据，使用平均需求
                return self.weekly_capacity
        except (IndexError, ValueError, TypeError):
            print(f"警告: 无法获取第{week}周的需求数据，使用默认产能值")
            return self.weekly_capacity

    def genetic_algorithm_optimize_orders(self, week, current_inventory, week_demand):
        """使用遗传算法优化单周订购计划"""
        # 问题：在满足需求和库存约束的前提下，最小化采购成本

        # 初始化种群
        population = self.initialize_population()

        for generation in range(self.ga_params['generations']):
            # 评估适应度
            fitness = [self.evaluate_fitness(individual, week_demand, current_inventory)
                      for individual in population]

            # 选择
            selected = self.selection(population, fitness)

            # 交叉
            offspring = self.crossover(selected)

            # 变异
            offspring = self.mutate(offspring)

            # 替换种群
            population = offspring

        # 找到最优个体
        best_idx = np.argmax([self.evaluate_fitness(ind, week_demand, current_inventory)
                             for ind in population])
        best_individual = population[best_idx]

        # 转换为订购计划格式
        weekly_orders = []
        for i, supplier in enumerate(self.selected_suppliers):
            if best_individual[i] > 0:
                weekly_orders.append({
                    'supplier_id': supplier['id'],
                    'material_type': supplier['material_type'],
                    'quantity': best_individual[i]
                })

        return weekly_orders

    def initialize_population(self):
        """初始化遗传算法种群"""
        population = []
        num_suppliers = len(self.selected_suppliers)

        for _ in range(self.ga_params['population_size']):
            # 为每个供应商生成随机订购量
            individual = []
            for supplier in self.selected_suppliers:
                # 基于供应商产能的随机订购量
                max_qty = supplier['capacity'] * 1.2  # 最多不超过产能的1.2倍
                min_qty = 0
                qty = random.uniform(min_qty, max_qty)
                individual.append(qty if random.random() > 0.7 else 0)  # 30%概率不订购
            population.append(individual)

        return population

    def evaluate_fitness(self, individual, demand, current_inventory):
        """评估个体适应度"""
        # 确保需求是数值类型
        if not isinstance(demand, (int, float)):
            try:
                demand = float(demand)
            except (ValueError, TypeError):
                return 0  # 无效需求，返回最低适应度

        # 计算总成本
        total_cost = 0
        total_supply = 0

        for i, qty in enumerate(individual):
            if qty <= 0:
                continue

            supplier = self.selected_suppliers[i]
            material_type = supplier['material_type']

            # 确保材料类型有效
            if material_type not in self.cost_coefficients or material_type not in self.material_conversion:
                continue

            # 计算成本 (基于C类的相对成本)
            total_cost += qty * self.cost_coefficients[material_type]

            # 计算等效供应量
            total_supply += qty * self.material_conversion[material_type]

        # 计算库存变化
        final_inventory = current_inventory + total_supply - demand

        # 惩罚库存不足
        if final_inventory < self.min_inventory:
            return 0  # 不可行解

        # 适应度是成本的倒数(最小化成本)加上库存合理性奖励
        inventory_ratio = final_inventory / (self.min_inventory * 2)  # 理想库存是最小库存的2倍
        inventory_reward = 1.0 - abs(1.0 - inventory_ratio)

        # 确保适应度为非负值
        fitness_value = (1.0 / (total_cost + 1e-6)) * (1.0 + inventory_reward)
        return max(0, fitness_value)  # 确保适应度不会为负

    def selection(self, population, fitness):
        """选择操作"""
        # 确保所有适应度值非负
        fitness = [max(0, f) for f in fitness]

        # 轮盘赌选择
        total_fitness = sum(fitness)

        if total_fitness == 0:
            # 如果所有适应度都是0，使用均匀分布选择
            probabilities = [1/len(population)] * len(population)
        else:
            # 计算概率，确保数值稳定性
            probabilities = [f / total_fitness for f in fitness]

            # 修正可能的数值误差导致的负概率
            probabilities = [max(0, p) for p in probabilities]

            # 重新归一化概率
            prob_sum = sum(probabilities)
            if prob_sum > 0:
                probabilities = [p / prob_sum for p in probabilities]
            else:
                probabilities = [1/len(population)] * len(population)

        selected = []
        for _ in range(len(population)):
            # 随机选择一个个体
            selected_idx = np.random.choice(len(population), p=probabilities)
            selected.append(population[selected_idx])

        return selected

    def crossover(self, population):
        """交叉操作"""
        offspring = []

        for i in range(0, len(population), 2):
            parent1 = population[i]
            parent2 = population[i+1] if i+1 < len(population) else population[0]

            if random.random() < self.ga_params['crossover_rate']:
                # 单点交叉
                point = random.randint(1, len(parent1) - 1)
                child1 = parent1[:point] + parent2[point:]
                child2 = parent2[:point] + parent1[point:]
                offspring.extend([child1, child2])
            else:
                # 不交叉，直接复制
                offspring.extend([parent1, parent2])

        return offspring[:len(population)]

    def mutate(self, population):
        """变异操作"""
        for i in range(len(population)):
            for j in range(len(population[i])):
                if random.random() < self.ga_params['mutation_rate']:
                    # 对第j个基因进行变异
                    supplier = self.selected_suppliers[j]
                    max_qty = supplier['capacity'] * 1.2
                    population[i][j] = random.uniform(0, max_qty)

        return population

    def optimize_transport_plan(self):
        """优化转运方案，最小化总损耗"""
        transport_plan = []

        # 为每周的订购计划制定转运方案
        for week_plan in self.order_plan:
            week = week_plan['week']
            orders = week_plan['orders']

            # 按供应商分组汇总
            supplier_orders = {}
            for order in orders:
                supplier_id = order['supplier_id']
                if supplier_id not in supplier_orders:
                    supplier_orders[supplier_id] = {
                        'total_quantity': 0,
                        'material_type': order['material_type']
                    }
                supplier_orders[supplier_id]['total_quantity'] += order['quantity']

            # 为每个供应商分配转运商
            weekly_transport = []
            for supplier_id, details in supplier_orders.items():
                # 找到损耗率最低的转运商
                best_transporter = self.find_best_transporter(details['material_type'])

                # 计算需要的运输次数
                total_qty = details['total_quantity']
                trips_needed = max(1, int(np.ceil(total_qty / self.transporter_capacity)))

                # 分配转运商
                weekly_transport.append({
                    'week': week,
                    'supplier_id': supplier_id,
                    'transporter_id': best_transporter['id'],
                    'material_type': details['material_type'],
                    'total_quantity': total_qty,
                    'trips': trips_needed,
                    'loss_rate': best_transporter['loss_rate']
                })

            transport_plan.append(weekly_transport)

        self.transport_plan = transport_plan
        return transport_plan

    def find_best_transporter(self, material_type):
        """为特定材料类型找到损耗率最低的转运商"""
        # 假设转运商数据中包含材料类型和对应的损耗率
        # 这里简化处理，选择总体损耗率最低的转运商
        sorted_transporters = self.transporter_data.sort_values('损耗率')
        return sorted_transporters.iloc[0].to_dict()

    def write_results_to_templates(self):
        """将结果写入附件A和B模板"""
        # 写入附件A (订购方案)
        self.write_order_plan_to_excel()

        # 写入附件B (转运方案)
        self.write_transport_plan_to_excel()

    def write_order_plan_to_excel(self):
        """将订购方案写入附件A"""
        # 创建一个副本，避免修改原始模板
        output_path = self.data_paths['output_a']

        # 读取模板并保留格式
        book = load_workbook(self.data_paths['template_a'])

        # 假设按材料类型分sheet存储
        material_sheets = {
            'A': 'A类原材料订购方案',
            'B': 'B类原材料订购方案',
            'C': 'C类原材料订购方案'
        }

        for material, sheet_name in material_sheets.items():
            if sheet_name not in book.sheetnames:
                continue

            sheet = book[sheet_name]
            # 找到数据开始的位置 (跳过前5行说明)
            start_row = 5

            # 收集该材料类型的所有订购数据
            material_orders = []
            for week_plan in self.order_plan:
                week = week_plan['week']
                for order in week_plan['orders']:
                    if order['material_type'] == material:
                        material_orders.append({
                            'week': week,
                            'supplier_id': order['supplier_id'],
                            'quantity': order['quantity']
                        })

            # 将数据转换为数据框并按周和供应商排序
            df = pd.DataFrame(material_orders)
            if not df.empty:
                df = df.pivot(index='supplier_id', columns='week', values='quantity').fillna(0)

                # 写入数据到Excel
                for r, (supplier_id, row) in enumerate(df.iterrows(), start=start_row + 1):
                    sheet.cell(row=r, column=1, value=supplier_id)  # 供应商ID
                    for c, week in enumerate(df.columns, start=2):
                        sheet.cell(row=r, column=c, value=row[week])

            # 确保保留最后一行的求和公式
            # (假设最后一行已经有公式，这里不做修改)

        # 保存结果
        book.save(output_path)
        print(f"订购方案已保存至: {output_path}")

    def write_transport_plan_to_excel(self):
        """将转运方案写入附件B"""
        # 创建一个副本，避免修改原始模板
        output_path = self.data_paths['output_b']

        # 读取模板并保留格式
        book = load_workbook(self.data_paths['template_b'])

        # 假设按材料类型分sheet存储
        material_sheets = {
            'A': 'A类原材料转运方案',
            'B': 'B类原材料转运方案',
            'C': 'C类原材料转运方案'
        }

        for material, sheet_name in material_sheets.items():
            if sheet_name not in book.sheetnames:
                continue

            sheet = book[sheet_name]
            # 找到数据开始的位置 (跳过前5行说明)
            start_row = 5

            # 收集该材料类型的所有转运数据
            material_transports = []
            for week_transports in self.transport_plan:
                for transport in week_transports:
                    if transport['material_type'] == material:
                        material_transports.append({
                            'week': transport['week'],
                            'supplier_id': transport['supplier_id'],
                            'transporter_id': transport['transporter_id'],
                            'quantity': transport['total_quantity'],
                            'trips': transport['trips']
                        })

            # 将数据转换为数据框并按周和供应商排序
            df = pd.DataFrame(material_transports)
            if not df.empty:
                # 这里需要根据模板的具体格式调整
                row_idx = start_row
                for _, row in df.iterrows():
                    row_idx += 1
                    sheet.cell(row=row_idx, column=1, value=row['week'])  # 周数
                    sheet.cell(row=row_idx, column=2, value=row['supplier_id'])  # 供应商ID
                    sheet.cell(row=row_idx, column=3, value=row['transporter_id'])  # 转运商ID
                    sheet.cell(row=row_idx, column=4, value=row['quantity'])  # 数量
                    sheet.cell(row=row_idx, column=5, value=row['trips'])  # 运输次数

            # 确保保留最后一行的求和公式
            # (假设最后一行已经有公式，这里不做修改)

        # 保存结果
        book.save(output_path)
        print(f"转运方案已保存至: {output_path}")

    def analyze_results(self):
        """分析订购方案和转运方案的实施效果"""
        # 计算总采购成本
        total_cost = 0
        material_counts = {'A': 0, 'B': 0, 'C': 0}

        for week_plan in self.order_plan:
            for order in week_plan['orders']:
                material_type = order['material_type']
                qty = order['quantity']
                if material_type in self.cost_coefficients:
                    total_cost += qty * self.cost_coefficients[material_type]
                    material_counts[material_type] += qty

        # 计算总损耗
        total_loss = 0
        total_transported = 0

        for week_transports in self.transport_plan:
            for transport in week_transports:
                qty = transport['total_quantity']
                loss_rate = transport['loss_rate'] / 100  # 转换为小数
                total_loss += qty * loss_rate
                total_transported += qty

        # 计算库存波动
        inventory_levels = [self.min_inventory]  # 初始库存
        for week_plan in self.order_plan:
            total_supply = sum(
                order['quantity'] * self.material_conversion[order['material_type']]
                for order in week_plan['orders']
                if order['material_type'] in self.material_conversion
            )
            next_inventory = max(
                self.min_inventory,
                inventory_levels[-1] + total_supply - week_plan['demand']
            )
            inventory_levels.append(next_inventory)

        inventory_variation = np.std(inventory_levels)

        # 输出分析结果
        print("\n===== 方案实施效果分析 =====")
        print(f"1. 供应商数量: {len(self.selected_suppliers)}")
        print(f"2. 总采购成本 (相对值): {total_cost:.2f}")
        print(f"3. 原材料采购量:")
        for material, count in material_counts.items():
            print(f"   - {material}类: {count:.2f} 立方米")
        print(f"4. 总运输量: {total_transported:.2f} 立方米")
        print(f"5. 总损耗量: {total_loss:.2f} 立方米")
        print(f"6. 损耗率: {total_loss / total_transported * 100:.2f}%" if total_transported > 0 else "6. 损耗率: N/A")
        print(f"7. 库存标准差 (波动程度): {inventory_variation:.2f}")

        # 绘制库存变化图
        plt.figure(figsize=(12, 6))
        plt.plot(range(len(inventory_levels)), inventory_levels, 'b-', marker='o')
        plt.axhline(y=self.min_inventory, color='r', linestyle='--', label='最低库存')
        plt.title('24周库存变化趋势')
        plt.xlabel('周数')
        plt.ylabel('库存水平 (等效立方米)')
        plt.grid(True)
        plt.legend()
        plt.savefig('inventory_trend.png')
        print(f"\n库存变化趋势图已保存为: inventory_trend.png")

        return {
            'supplier_count': len(self.selected_suppliers),
            'total_cost': total_cost,
            'material_counts': material_counts,
            'total_transported': total_transported,
            'total_loss': total_loss,
            'loss_rate': total_loss / total_transported * 100 if total_transported > 0 else 0,
            'inventory_variation': inventory_variation
        }

if __name__ == "__main__":
    # 定义数据文件路径
    data_paths = {
        'topsis_results': '供应商TOPSIS评价结果.xlsx',
        'supplier_data': '附件1.xlsx',
        'transporter_data': '附件2.xlsx',
        'demand_forecast': '期望供货量预测结果.xlsx',
        'template_a': '附件A.xlsx',
        'template_b': '附件B.xlsx',
        'output_a': '附件A_结果.xlsx',
        'output_b': '附件B_结果.xlsx'
    }

    # 根据实际Excel文件的sheet名称配置映射关系
    # 请根据您的实际文件修改以下映射
    sheet_mapping = {
        'Sheet1': 'A',  # 例如：Sheet1对应A类材料
        'Sheet2': 'B',  # 例如：Sheet2对应B类材料
        'Sheet3': 'C',  # 例如：Sheet3对应C类材料
        # 可以根据实际情况添加更多映射
    }

    # 验证文件是否存在
    for name, path in data_paths.items():
        if name not in ['output_a', 'output_b'] and not os.path.exists(path):
            print(f"警告: 未找到文件 {path}，请检查路径是否正确")

    # 创建优化器实例，传入sheet映射
    optimizer = SupplyChainOptimizer(data_paths, sheet_mapping)

    # 执行供应商选择
    print("===== 开始供应商选择 =====")
    optimizer.select_suppliers()

    # 优化订购计划
    print("\n===== 开始订购计划优化 =====")
    optimizer.optimize_order_plan()

    # 优化转运计划
    print("\n===== 开始转运计划优化 =====")
    optimizer.optimize_transport_plan()

    # 写入结果到模板
    print("\n===== 写入结果到Excel =====")
    optimizer.write_results_to_templates()

    # 分析结果
    optimizer.analyze_results()

    print("\n===== 所有优化完成 =====")
    print(f"订购方案结果: {data_paths['output_a']}")
    print(f"转运方案结果: {data_paths['output_b']}")


===== 开始供应商选择 =====
选择的供应商数量: 402
总等效产能: 670000.00
总需求: 676800.00

===== 开始订购计划优化 =====
警告: 无法获取第1周的需求数据，使用默认产能值
警告: 无法获取第2周的需求数据，使用默认产能值
警告: 无法获取第3周的需求数据，使用默认产能值
警告: 无法获取第4周的需求数据，使用默认产能值
警告: 无法获取第5周的需求数据，使用默认产能值
警告: 无法获取第6周的需求数据，使用默认产能值
警告: 无法获取第7周的需求数据，使用默认产能值
警告: 无法获取第8周的需求数据，使用默认产能值
警告: 无法获取第9周的需求数据，使用默认产能值
警告: 无法获取第10周的需求数据，使用默认产能值
警告: 无法获取第11周的需求数据，使用默认产能值
警告: 无法获取第12周的需求数据，使用默认产能值
警告: 无法获取第13周的需求数据，使用默认产能值
警告: 无法获取第14周的需求数据，使用默认产能值
警告: 无法获取第15周的需求数据，使用默认产能值
警告: 无法获取第16周的需求数据，使用默认产能值
警告: 无法获取第17周的需求数据，使用默认产能值
警告: 无法获取第18周的需求数据，使用默认产能值
警告: 无法获取第19周的需求数据，使用默认产能值
警告: 无法获取第20周的需求数据，使用默认产能值
警告: 无法获取第21周的需求数据，使用默认产能值
警告: 无法获取第22周的需求数据，使用默认产能值
警告: 无法获取第23周的需求数据，使用默认产能值
警告: 无法获取第24周的需求数据，使用默认产能值

===== 开始转运计划优化 =====


KeyError: '损耗率'